In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf

In [ ]:
# Taxi-Datensatz von https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# Data Dictionary: https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf
df = pd.read_parquet('daten/yellow_tripdata_2024-04.parquet')

In [ ]:
# Überblick
df.describe().round(3)

In [ ]:
# Features (=neue Variablen bzw. Werte), die wir aus bestehenden ableiten
df["duration"]     = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
df["duration_min"] = df["duration"].dt.total_seconds() / 60
df["avg_speed"]    = df["trip_distance"] / (df['duration_min'] / 60)
df["tip_pct"]      = df["tip_amount"] / df["total_amount"]

In [ ]:
# Wir erzeugen einen neuen Dataframe, bei dem wir unerwünschte Werte ausfiltern
# Wir haben die Grenzen der Datenfilterung willkürlich gewählt: auf Basis unseres Vorwissens
# bzw. unserer Fragestellung. Es gibt selten eindeutige Kriterien dafür.
df2 = df.query((
    "total_amount > 1 and "
    "tolls_amount < 100 and "
    "trip_distance > 0.1 and trip_distance < 100 and "
    "duration_min > 1 and duration_min < 180 and "
    "avg_speed > 1 and avg_speed < 75 and "
    'tpep_pickup_datetime > "2024-04-01 00:00:00" and '
    "tpep_dropoff_datetime > tpep_pickup_datetime"
))

In [ ]:
df2.shape
# Wir haben inzwischen über 100.000 Datensätze herausgefiltert. Das entspricht ca. 3% der Datensätze.

In [ ]:
# Da folgende Berechnungen teils recht lange dauern können,
# nehmen wir nur ein Sample des gesamten Datensatzes
df3 = df2.sample(10000, random_state=1234)

In [ ]:
# Wir stellen ein Modell auf, bei der wir Entfernung, Dauer und Trinkgeld mit einfließen lassen.
model = smf.ols("total_amount ~ trip_distance + duration_min + tip_amount",
                data=df3).fit()

In [ ]:
model.summary()

In [ ]:
# Residuen als Scatterplot in Abhängigkeit vom verhergesagten Wert
sns.scatterplot(x=model.fittedvalues, y=model.resid, alpha=0.1)

In [ ]:
# y vs ŷ Plot (echte vs. vorhergesagte Daten)
sns.scatterplot(x=df3["total_amount"], y=model.fittedvalues, alpha=0.1)

## Einfluß einzelner Datensätze

Wir schauen uns nun an, wie stark einzelne Datensätze das Modell beeinflussen.
Optimalerweise gehen wir davon aus, dass jeder Punkt mit etwa demselben Gewicht
in das Modell einfließt, d.h. denselben Einfluss hat.
In der Realität ist das leider nicht so.

Mit Hilfe der `get_influence()` Funktion und der darin enthaltenen Metriken können
wir abschätzen, wie stark Punkte die Modellkoeffizienten beeinflussen.

In [ ]:
influence = model.get_influence()

**Cook’s Distance:**
Ein Maß dafür, wie stark eine einzelne Beobachtung die geschätzten Regressionskoeffizienten beeinflusst. Große Werte deuten darauf hin, dass der Punkt einen unverhältnismäßig großen Einfluss auf das Modell hat. Als grobe Daumenregel gelten Werte größer als 1 als auffällig; bei größeren Datensätzen können auch Werte über
$4 \over n$ (mit $n$ = Anzahl der Beobachtungen) bereits relevant sein.

**Leverage (Hebelwerte):**
Beschreibt, wie weit eine Beobachtung in den erklärenden Variablen vom Durchschnitt entfernt ist. Punkte mit hohem Leverage liegen „am Rand“ des Datenraums und haben potenziell großen Einfluss auf die Regression. Ein häufiger Richtwert ist $2p \over n$ oder $3p \over n$ (mit $p$ = Anzahl der Parameter inklusive Intercept) als Schwelle für auffällige Werte.

**Studentized Residuals:**
Standardisierte Residuen, die die Streuung der Fehler berücksichtigen und daher besser vergleichbar sind als einfache Residuen. Sie zeigen, wie stark eine Beobachtung vom Modell vorhergesagten Wert abweicht. Werte größer als etwa ±3 gelten als auffällig, Werte über ±4 als stark ungewöhnlich und potenzielle Ausreißer.


In [ ]:
cooksd = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag;
studresid = influence.resid_studentized_external;

In [ ]:
sns.histplot(studresid)
# Auf Basis des Histogramms sehen wir, dass sehr viele der Residuen im gewünschten Wert im Bereich sind.
# Allerdings gibt es offensichtlich grobe Ausreißer, die bis auf +30 gehen.

In [ ]:
(np.abs(studresid) > 3).sum()
# viele >3 was bei 10000 Datensätzen nicht völlig ungewöhnlich ist

In [ ]:
(np.abs(studresid) > 4).sum()

In [ ]:
from scipy.stats import norm
# Wenn die Variable in exakt normal verteilt, würden wir uns so viele Werte erwarten:

# 27 Werte größer 3
print(round(2*norm.cdf(-3)*10000), "mal |Wert| > 3")
# Erklärung: norm.cdf(-3): %-Anteil der Werte kleiner -3;
# da Normalverteilung symmetrisch, ist derselbe %-Anteil > 3, deshalb einfach *2
# *10000: so viele Datensätze haben wir

# max. 1 Wert größer 4
print(round(2*norm.cdf(-4)*10000), "mal |Wert| > 4")

# Das heißt, wir sehen deutlich, dass unsere Werte nicht der Erwartung entsprechen.

In [ ]:
# Dargestellt als Diagramm
# >4 hatte recht viele Punkte mitgenommen, Plot für >5
outlier = np.abs(studresid) > 5
normal = ~outlier

sns.scatterplot(x=df3["total_amount"].iloc[normal],
                y=model.fittedvalues[normal], alpha=0.1)
sns.scatterplot(x=df3["total_amount"].iloc[outlier],
                y=model.fittedvalues[outlier], alpha=0.1)

In [ ]:
# Cook's Distance hat Punkte bis 5!
# Wenn wir hineinzoomen, sieht es so aus:
sns.histplot(cooksd[cooksd<0.001])

In [ ]:
n = model.nobs

# Empfohlene Grenze 4/n
(cooksd > 4/n).sum()

# Das sind wirklich viele Punkte (589). So viele wollen wir sicher nicht entfernen

In [ ]:
# Nochmal als Graph -- es zeigt, dass wir viele Punkte wegnehmen würden
# Sinnvollerweise sollten wir das eingrenzen und nicht gleich 4/n als Grenze ziehen
outlier = cooksd > 4/n
super_outlier = cooksd > 1
normal = ~outlier

sns.scatterplot(x=df3["total_amount"].iloc[normal],
                y=model.fittedvalues[normal], alpha=0.1)
sns.scatterplot(x=df3["total_amount"].iloc[outlier],
                y=model.fittedvalues[outlier], alpha=0.1)
sns.scatterplot(x=df3["total_amount"].iloc[super_outlier],
                y=model.fittedvalues[super_outlier], alpha=0.5)

In [ ]:
sns.histplot(leverage)

# Wieder eine ähnliche Verteilung bei der Leverage.
# Die meisten Punkte sind okay, nur wenige haben einen zu hohen Wert.

In [ ]:
# n .. Anzahl der Datensätze
# p .. Anzahl der Modellparameter
p = model.params.size

# Grenze: 2*p/n

In [ ]:
outlier = leverage > 2*p/n
normal = ~outlier

sns.scatterplot(x=df3["total_amount"].iloc[normal],
                y=model.fittedvalues[normal], alpha=0.1)
sns.scatterplot(x=df3["total_amount"].iloc[outlier],
                y=model.fittedvalues[outlier], alpha=0.1)

# Auch hier ist die Grenze zu eng gezogen.
# -> Wir müssen die Kriterien ein wenig aufweichen.

In [ ]:
# Idee: anstatt die Grenzen auf Basis der Literaturempfehlung zu nehmen,
# nehmen wir als Grenze den jeweils 50-zig-höchsten Wert und scheiden dann die entsprechenden Punkte aus.

limit_cooksd = sorted(cooksd, reverse=True)[50]
limit_leverage = sorted(leverage, reverse=True)[50]
limit_tresid = sorted(np.abs(studresid), reverse=True)[50]

In [ ]:
# Nur wenn alle 3 Kriterien zutreffen, scheiden wir einen Punkt aus
df4 = df3[(cooksd < limit_cooksd) |
          (leverage < limit_leverage) |
          (np.abs(studresid) < limit_tresid)]

In [ ]:
df4.info()
# nur 17 Punkte weniger

In [ ]:
# Wir setzen ein neues Modell mit dem reduzierten Datensatz auf
model2 = smf.ols("total_amount ~ trip_distance + duration_min + tip_amount",
                data=df4).fit()

In [ ]:
model2.summary()

In [ ]:
# Vergleich der Koeffizienten: ein deutlicher Unterschied
pd.DataFrame({"coef1": model.params,
              "coef2": model2.params,
              "stderr": model.bse,
              "stderr2": model2.bse}).round(3)

In [ ]:
# Vergleich der Konfidenzintervalle
ci = pd.concat([model.conf_int(), model2.conf_int()], axis=1)
ci.columns = ['m1_lower', 'm1_upper', 'm2_lower', 'm2_upper']
ci.round(2)

In [ ]:
# Darstellung von y vs ŷ mit optimaler Linie (y=ŷ)
# Die stark abweichenden Punkte sind deutlich erkennbar
diag = sns.scatterplot(y=model.fittedvalues, x=df3["total_amount"], alpha=0.3)
x_vals = np.array(diag.get_xlim())
y_vals = 0 + 1 * x_vals
diag.plot(x_vals, y_vals, color='green')

In [ ]:
# Im zweiten Modell fehlen diese, obwohl der Punkt rechts unten immer
# noch hervorsticht. Der hat zwar ein hohes Studentized Residual und
# eine hohe Cook's Distance, aber (unerwartet) eine kleine Leverage

diag = sns.scatterplot(y=model2.fittedvalues, x=df4["total_amount"])
x_vals = np.array(diag.get_xlim())
y_vals = 0 + 1 * x_vals
diag.plot(x_vals, y_vals, color='green')

# Auswirkung einer Log-Transformation

In [ ]:
df3["duration_min"].hist(bins=30)

In [ ]:
np.log(df3["duration_min"]).hist(bins=30)
# Nach der log-Transformation ist die Schieflastigkeit "verschwunden"
# Eine Log-Transformation kann also helfen, mit schieflastigen Verteilungen umzugehen,
# ohne die extremen Werte ausschließen zu müssen.

In [ ]:
sns.scatterplot(x="duration_min", y="total_amount",
               data=df3, alpha=0.2)

In [ ]:
sns.scatterplot(x=np.log(df3["duration_min"]),
                y=np.log(df3["total_amount"]), alpha=0.2)

# Effektgröße

**Standardisierte Regressionskoeffizienten (std_beta):**
Geben an, wie stark sich die Zielvariable (in Standardabweichungen) verändert, wenn sich eine X-Variable um eine Standardabweichung ändert. Dadurch sind Effekte zwischen Variablen direkt vergleichbar. Als grobe Orientierung gelten Beträge von etwa 0.1 als klein, 0.3 als mittel und 0.5 oder größer als stark, wobei der Kontext entscheidend bleibt.

**Partielles Eta-Quadrat (partial_eta²):**
Misst den Anteil der erklärten Varianz eines Effekts relativ zur verbleibenden (Fehler-)Varianz in Modellen wie ANOVA. Werte liegen zwischen 0 und 1. Häufig genutzte Richtwerte sind etwa 0.01 (klein), 0.06 (mittel) und 0.14 (groß). Diese Schwellen sind nur Daumenregeln und sollten immer im Kontext des jeweiligen Anwendungsfelds interpretiert werden.

In [ ]:
def effect_sizes(model, sort_by="std_beta"):
    X = pd.DataFrame(model.model.exog, columns=model.model.exog_names)
    y = model.model.endog
    eff = pd.DataFrame({
            "coef": model.params,
            "std_beta": model.params * X.std() / y.std(),
            "partial_eta2": model.tvalues**2 / (model.tvalues**2 + model.df_resid),
            "p_value": model.pvalues,
        })
    eff = eff.drop("Intercept")
    eff = eff.sort_values(by=sort_by, key=lambda k: abs(k), ascending=False)
    return eff

In [ ]:
effect_sizes(model2)
# Wir haben es nur mit sehr großen Effekten zu tun :o)

In [ ]:
# Zum Vergleich: wenn wir die Einheiten der Daten ändern
df4["duration_hour"] = df4["duration_min"] / 60
df4["distance_cm"] = df4["trip_distance"] * 160934

In [ ]:
model3 = smf.ols("total_amount ~ distance_cm + duration_hour + tip_amount",
                data=df4).fit()

In [ ]:
model3.summary()
# dann sehen die Koeffizienten deutlich anders aus

In [ ]:
model2.params

In [ ]:
model3.params

In [ ]:
effect_sizes(model3)
# aber der Effekt bleibt derselbe, weil wir die Daten im engeren Sinn ja nicht verändern:
# eine weite Entfernung bleibt immer noch weit, eine große Dauer bleibt immer noch groß.
# -> Die Relation der Daten zueinander und untereinander verändert sich nicht.
# Es verändert sich nur ein Multiplikator, der aber auf den Effekt, auf die Information,
# die in der Variable steckt, keinen Einfluss hat.